In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

# 1) 데이터 생성
np.random.seed(42)

chars = list("abcdefghijklmnopqrstuvwxyz")
PAD = "_"
SOS = "^"
EOS = "$"

vocab = [PAD, SOS, EOS] + chars
char2idx = {c:i for i,c in enumerate(vocab)}
idx2char = {i:c for c,i in char2idx.items()}

def make_samples(n=5000, min_len=3, max_len=8):
    X, Y = [], []
    for _ in range(n):
        L = np.random.randint(min_len, max_len+1)
        s = "".join(np.random.choice(chars, size=L))
        t = s[::-1]
        X.append(s)
        Y.append(t)
    return X, Y

X_text, Y_text = make_samples(n=6000)
max_in = max(len(s) for s in X_text) + 2   # SOS, EOS 포함
max_out = max(len(s) for s in Y_text) + 2

def vectorize(src_list, tgt_list):
    encoder_in = np.zeros((len(src_list), max_in, len(vocab)), dtype="float32")
    decoder_in = np.zeros((len(src_list), max_out, len(vocab)), dtype="float32")
    decoder_out = np.zeros((len(src_list), max_out, len(vocab)), dtype="float32")

    for i, (src, tgt) in enumerate(zip(src_list, tgt_list)):
        src_seq = SOS + src + EOS
        tgt_seq = SOS + tgt + EOS

        # encoder input
        for t, ch in enumerate(src_seq.ljust(max_in, PAD)):
            encoder_in[i, t, char2idx[ch]] = 1.0

        # decoder input / output (teacher forcing)
        tgt_seq_pad = tgt_seq.ljust(max_out, PAD)
        for t, ch in enumerate(tgt_seq_pad):
            decoder_in[i, t, char2idx[ch]] = 1.0
            if t > 0:
                decoder_out[i, t-1, char2idx[ch]] = 1.0

    return encoder_in, decoder_in, decoder_out

enc_in, dec_in, dec_out = vectorize(X_text, Y_text)

# 2) 모델 정의 (Encoder-Decoder)
latent_dim = 128

encoder_inputs = Input(shape=(None, len(vocab)))
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None, len(vocab)))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(len(vocab), activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# 3) 학습
history = model.fit([enc_in, dec_in], dec_out, batch_size=64, epochs=15, validation_split=0.2)

# 4) 추론용 모델 (inference)
encoder_model = Model(encoder_inputs, encoder_states)

dec_state_input_h = Input(shape=(latent_dim,))
dec_state_input_c = Input(shape=(latent_dim,))
dec_states_inputs = [dec_state_input_h, dec_state_input_c]

dec_outputs, state_h2, state_c2 = decoder_lstm(decoder_inputs, initial_state=dec_states_inputs)
dec_states = [state_h2, state_c2]
dec_outputs = decoder_dense(dec_outputs)

decoder_model = Model([decoder_inputs] + dec_states_inputs, [dec_outputs] + dec_states)

def decode_sequence(input_seq_onehot):
    states_value = encoder_model.predict(input_seq_onehot, verbose=0)

    # 시작 토큰
    target_seq = np.zeros((1, 1, len(vocab)), dtype="float32")
    target_seq[0, 0, char2idx[SOS]] = 1.0

    decoded = []
    for _ in range(max_out):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_idx = int(np.argmax(output_tokens[0, -1, :]))
        sampled_char = idx2char[sampled_idx]

        if sampled_char == EOS:
            break
        if sampled_char != PAD and sampled_char != SOS:
            decoded.append(sampled_char)

        target_seq = np.zeros((1, 1, len(vocab)), dtype="float32")
        target_seq[0, 0, sampled_idx] = 1.0
        states_value = [h, c]

    return "".join(decoded)

# 5) 테스트 출력
def to_onehot_src(s):
    src_seq = (SOS + s + EOS).ljust(max_in, PAD)
    x = np.zeros((1, max_in, len(vocab)), dtype="float32")
    for t, ch in enumerate(src_seq):
        x[0, t, char2idx[ch]] = 1.0
    return x

tests = ["cat", "hello", "apple", "mouse", "train"]
for s in tests:
    pred = decode_sequence(to_onehot_src(s))
    print(f"IN : {s:>8} | PRED: {pred:>8} | TRUE: {s[::-1]:>8}")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 29)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 29)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     80,896 │ input_layer[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     80,896 │ input_layer_1[0]… │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 29)  │      3,741 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 165,533 (646.61 KB)

 Trainable params: 165,533 (646.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3783 - loss: 2.2998 - val_accuracy: 0.4640 - val_loss: 2.0139
Epoch 2/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4796 - loss: 1.9035 - val_accuracy: 0.4870 - val_loss: 1.8378
Epoch 3/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4966 - loss: 1.8479 - val_accuracy: 0.5040 - val_loss: 1.7823
Epoch 4/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5342 - loss: 1.6771 - val_accuracy: 0.5489 - val_loss: 1.5833
Epoch 5/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5676 - loss: 1.4848 - val_accuracy: 0.5828 - val_loss: 1.3847
Epoch 6/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5979 - loss: 1.3047 - val_accuracy: 0.6047 - val_loss: 1.2562
Epoch 7/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6221 - loss: 1.1805 - val_accuracy: 0.6332 - val_loss: 1.1419
Epoch 8/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6587 - loss: 1.0581 - val_accuracy: 0.6678 - v

In [5]:
# 사용자 입력 받아서 뒤집기 결과 출력
while True:
    user_input = input("뒤집을 문자열 입력 (종료: exit): ")

    if user_input.lower() == "exit":
        print("종료합니다 👋")
        break

    # 입력 문자열 one-hot 변환
    x = to_onehot_src(user_input)

    # 예측
    pred = decode_sequence(x)

    print(f"입력: {user_input}")
    print(f"모델 출력: {pred}")
    print(f"정답: {user_input[::-1]}")
    print("-" * 40)


입력: hiyou
모델 출력: uoyih
정답: uoyih
----------------------------------------
종료합니다 👋
